# Lab 21: Time Series Forecasting — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 135 min Core (setup and README included; no optional parts)

---

**Learning Objectives:**
- Diagnose an analyst's ARIMA forecasting pipeline for CPI that has three errors, and say what each one does to the forecast
- Repair the pipeline, judge the repaired model's residuals with a test you have seen reject, and say what its forecast does differently
- Fit a GARCH(1,1) to S&P 500 returns, and turn its two parameters into the half-life of a volatility shock
- Write an expanding-window backtest that scores a forecaster by MASE at many forecast origins
- Ship a module that compares forecasters by backtest, and use it to judge the repaired model against a seasonal naive forecast

**Dataset:** CPIAUCNS from FRED (the consumer price index, not seasonally adjusted, monthly from 2000), pulled with your FRED key, stopped at June 2026 and taken as FRED published it on 9 October 2026, so that every number below stays the same all term; and the S&P 500's daily closes from 2000 to 2024, from a saved copy in the course repository.

**Prerequisites:** your FRED key, saved as the Colab secret `FRED_API_KEY` (Lab 2); ADF and KPSS (Lab 20).

**Format:** Part 1 shows you an analyst's forecasting pipeline with three errors and asks what happened. In Part 2 you repair it, filling in blanks. Part 3 is run-and-read, with two blanks. Write it yourself and Part 4 are yours. Write your answers in the text cell under each set of questions.

**Deliverable:** your written answers (Parts 1 to 4), the blanks you fill in Parts 2 and 3, the required `backtest_expanding_window()` function, and `src/forecast_evaluation.py`, a module you finish with your function, save and upload.

**Time:** about 135 minutes in all, at the pace of a first reading: setup 8, Part 1 30, Part 2 24, Part 3 15, Write it yourself 34, Part 4 12, and the README and GitHub submission 12. Everything in this lab is required; there is no optional section.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Part 2:** three blanks, in two cells, that repair the analyst's pipeline.
> - **Part 3:** two blanks, the persistence of a volatility shock and its half-life.
> - **Write it yourself (required, after Part 3):** `backtest_expanding_window()`, a function that scores a forecaster at every forecast origin of a series and returns a table, about twelve lines in an empty cell. A check cell tells you when it is right. In Part 4 you paste it, unchanged, into the module.
>
> **You write, in text cells:** three answers in Part 1 and two each in Parts 2, 3 and 4.
>
> **You run and read:** everything else. That is the FRED pull and the saved S&P 500 file, the analyst's three cells and the random walk beside them, the repaired forecast and its score on one year, the GARCH fit and its volatility chart, and the rest of Part 4's module, `forecast_evaluation.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints, and Write it yourself turns Part 2's score on one year into a score on eleven years.

## Setup

The first line of the setup cell installs `fredapi` (FRED's interface), statsmodels 0.14 or 0.15 and
`arch` 8 (GARCH, for Part 3): the expected numbers in this lab were produced on those versions. The
cell prints them. If it stops with an `ImportError`, or shows other versions, use **Runtime →
Restart session** and run this cell again. The `warnings.filterwarnings` line hides library
notices; one of them is about the analyst's model in Part 1, which statsmodels warns did not
settle, and Part 1's first question is about that model.

The second cell reads your FRED key from Colab Secrets, as in Lab 2. If Colab asks whether this
notebook may read `FRED_API_KEY`, allow it. It also sets `END`, the last month this lab uses, and
`VINTAGE`, the day whose published data it asks for. FRED adds a month every few weeks and revises
earlier ones; asking for CPI up to June 2026 as FRED had it on 9 October 2026 gives you the numbers
in the expected outputs, whenever you run the lab.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries
!pip install "fredapi>=0.5,<0.6" "statsmodels>=0.14,<0.16" "arch>=8,<9" -q

import hashlib
import io
import urllib.request
import warnings
from pathlib import Path

import arch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels
from arch import arch_model
from fredapi import Fred
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

Path("src").mkdir(exist_ok=True)     # Part 4 writes its module into this folder
warnings.filterwarnings("ignore")    # library notices; Part 1 asks about the one its model gives
print(f"statsmodels {statsmodels.__version__} (this lab needs 0.14 or 0.15), arch {arch.__version__} (needs 8)")
print("Setup complete.")

In [ ]:
# GIVEN — run as-is
# Setup, continued: your FRED key, from Colab Secrets, and the last month this lab uses
# Your FRED key comes from Colab's secret store, never from this cell:
#   left sidebar -> key icon (Secrets) -> + Add new secret
#   Name: FRED_API_KEY    Value: your 32-character key    Notebook access: ON
# Free key: https://fredaccount.stlouisfed.org/apikeys
# You set this ONCE; every notebook on the same Google account then finds it.
FRED_API_KEY = ""
# try/except: try the Colab secret; outside Colab, fall back to an environment variable
try:
    from google.colab import userdata
    FRED_API_KEY = userdata.get("FRED_API_KEY") or ""
except Exception:
    import os
    FRED_API_KEY = os.environ.get("FRED_API_KEY", "")

if not FRED_API_KEY:
    # No secret set. getpass hides what you type, so the key stays out of this
    # notebook and out of its saved output either way.
    import getpass
    FRED_API_KEY = getpass.getpass("FRED API key (hidden as you type): ").strip()

if len(FRED_API_KEY) < 20:
    raise ValueError(
        "No FRED API key found. Get one free at "
        "https://fredaccount.stlouisfed.org/apikeys, then add it as a Colab "
        "secret named FRED_API_KEY with Notebook access switched ON -- missing "
        "that switch is the most common cause of this message."
    )

fred = Fred(api_key=FRED_API_KEY)
END = "2026-06-01"                   # the last month used
VINTAGE = "2026-10-09"               # the data as FRED published them that day, so later revisions change nothing
print(f"FRED ready. CPI stops at {END}, as published on {VINTAGE}.")

---

## Part 1: DIAGNOSE — Find 3 Errors in This ARIMA Pipeline (30 min)

An analyst forecasts U.S. consumer prices two years ahead with the Box-Jenkins cycle from class:
choose the number of differences d with a unit-root test, fit an ARIMA, check its residuals, and
forecast. The pipeline has three errors, one in each of the analyst's three cells. Run them, and the
given cells beside them, then diagnose.

**Reading the code.** The pull is Lab 20's: `observation_end=END` stops CPI at June 2026,
`realtime_start` and `realtime_end` ask for the values published on `VINTAGE`, and `.asfreq("MS")`
shows a missing month as `NaN`. statsmodels will not fit a series with a gap, so
`.interpolate(method="time")` fills it on the line between its two neighbours.

In [ ]:
# GIVEN — run as-is
# CPI (CPIAUCNS: all urban consumers, 1982-84 = 100, not seasonally adjusted), monthly, January 2000 to END
cpi = fred.get_series("CPIAUCNS", observation_start="2000-01-01", observation_end=END,
                      realtime_start=VINTAGE, realtime_end=VINTAGE)
cpi = cpi.dropna()
cpi.index = pd.DatetimeIndex(cpi.index)
cpi = cpi.asfreq("MS")                # one row per month start
missing = cpi.index[cpi.isna()]
print(f"missing months: {[str(date.date()) for date in missing]}")
cpi = cpi.interpolate(method="time")  # each gap filled on the line between its two neighbours
cpi.name = "CPI"
print(f"{cpi.index[0].date()} to {cpi.index[-1].date()}: {len(cpi)} months; "
      f"October 2025 filled in as {cpi['2025-10-01']:.3f}")

**Expected output:**
```
missing months: ['2025-10-01']
2000-01-01 to 2026-06-01: 318 months; October 2025 filled in as 324.467
```
October 2025 is missing from the official series: BLS published no CPI for that month. The
filled-in value is a choice this lab makes, not a measurement.

**Reading the code.** `adfuller` is Lab 20's ADF test; `[:2]` keeps its first two results, the
statistic and the p-value. `d = 0 if stationary else 1` sets d to 0 when `stationary` is `True`.
`ARIMA(series, order=(p, d, q)).fit()` estimates a model with p AR terms, d differences and q MA
terms. With d = 0 it also has `const`, the mean the series returns to in the long run, and it must
be stationary, so statsmodels keeps the AR estimates inside the region where shocks die out (for
an AR(2) part, one condition is ar.L1 + ar.L2 < 1). `sigma2` is the variance of the shocks.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 1: choose d with an ADF test, then fit ARIMA(2, d, 1) to CPI
adf_stat, adf_p = adfuller(cpi, autolag="AIC", regression="ct")[:2]
stationary = adf_p > 0.05
d = 0 if stationary else 1
print(f"ADF statistic {adf_stat:.4f}   p-value {adf_p:.4f}   stationary: {stationary}   so d = {d}")

analyst = ARIMA(cpi, order=(2, d, 1)).fit()
print(analyst.params.round(4).to_string())

**Expected output:**
```
ADF statistic -0.8907   p-value 0.9571   stationary: True   so d = 0
const     243.3900
ar.L1       1.9999
ar.L2      -1.0000
ma.L1      -1.0000
sigma2      0.8285
```

**Reading the code.** A model's residuals, `.resid`, are CPI minus the model's prediction of each
month. The **Ljung-Box** test from class has the null hypothesis "the first k autocorrelations of
the residuals are all zero", so a small p-value says the model left structure behind.
`acorr_ljungbox(resid, lags=[12, 24], return_df=True)` tests up to lag 12 and up to lag 24, one row
each. `plot_acf(resid, lags=36)` draws the residual autocorrelations with a shaded 95% band.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 2: check that the residuals are white noise, with a Ljung-Box test at lags 12 and 24
lb = acorr_ljungbox(analyst.resid, lags=[12, 24], return_df=True)
print(lb.round(4).to_string())
plot_acf(analyst.resid, lags=36)
plt.show()

**Expected output:**
```
    lb_stat  lb_pvalue
12   0.3412        1.0
24   0.7186        1.0
```
then an autocorrelation chart.

**A model whose residuals you know are not white noise.** A random walk, ARIMA(0, 1, 0), forecasts
each month as the month before, so its residuals are CPI's monthly changes, `cpi.diff()`. The next
cell measures their correlation a year apart (`.autocorr(lag=12)`), then runs the analyst's Step 2
on the random walk.

In [ ]:
# GIVEN — run as-is
# The analyst's Step 2 on a random walk, whose residuals are CPI's monthly changes
random_walk = ARIMA(cpi, order=(0, 1, 0)).fit()
print(f"autocorrelation of CPI's monthly changes, 12 months apart: {cpi.diff().autocorr(lag=12):.4f}")
print(acorr_ljungbox(random_walk.resid, lags=[12, 24], return_df=True).round(4).to_string())

**Expected output:**
```
autocorrelation of CPI's monthly changes, 12 months apart: 0.3622
    lb_stat  lb_pvalue
12   0.0591        1.0
24   0.1242        1.0
```

**Reading the code.** Two given helpers. `plot_forecast` draws the last five years of CPI, a
forecast (`.predicted_mean`) and its 95% interval (`.conf_int(alpha=0.05)`, a lower and an upper
column). `month_by_month` sets the forecast's change in each month of 2027 beside CPI's average
change in the same calendar month, 2000 to 2025 (`.groupby(...month)`, as in Lab 20). In the
analyst's Step 3, `.get_forecast(steps=24)` forecasts the 24 months after June 2026.

In [ ]:
# GIVEN — run as-is
# Two helpers for a forecast: a chart, and its monthly changes beside CPI's usual ones
def plot_forecast(forecast, label):
    """The last five years of CPI, the forecast and its 95% interval."""
    band = forecast.conf_int(alpha=0.05)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(cpi["2021-07":], label="CPI")
    ax.plot(forecast.predicted_mean, label=label)
    ax.fill_between(band.index, band.iloc[:, 0], band.iloc[:, 1], alpha=0.2, label="95% interval")
    ax.legend()
    plt.show()


def month_by_month(forecast):
    """CPI's average change in each calendar month, 2000 to 2025, beside the forecast's change in 2027."""
    changes = cpi.diff()["2000":"2025"]
    usual = changes.groupby(changes.index.month).mean()
    ahead = forecast.predicted_mean.diff()["2027"]
    print("month   CPI's average change   forecast change in 2027")
    for month in range(1, 13):
        print(f"{month:>5} {usual[month]:>22.3f} {ahead.iloc[month - 1]:>25.3f}")

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 3: forecast the 24 months after June 2026, with a 95% interval
analyst_forecast = analyst.get_forecast(steps=24)
plot_forecast(analyst_forecast, "the analyst's forecast")
month_by_month(analyst_forecast)

**Expected output:** a chart, then
```
month   CPI's average change   forecast change in 2027
    1                  1.003                     0.460
    2                  1.069                     0.459
    3                  1.239                     0.457
    4                  0.834                     0.456
    5                  0.759                     0.454
    6                  0.755                     0.453
    7                  0.232                     0.451
    8                  0.332                     0.450
    9                  0.484                     0.448
   10                  0.063                     0.446
   11                 -0.396                     0.445
   12                 -0.364                     0.443
```

### YOUR DIAGNOSIS

1. Step 1. What does an ADF p-value of 0.9571 say about CPI, and what did the analyst's code conclude? What do ar.L1 + ar.L2 and `const` say the fitted model is doing?
2. Step 2. Both models pass the analyst's check with p = 1.0000, although the random walk's residuals are correlated at 0.36 a year apart. Run `print(analyst.resid.head(3), random_walk.resid.head(3))` in a new cell. The first residual of each model is the difference between what and what, and what does one residual that size do to the test?
3. Step 3. What does the analyst's model leave out, as the table shows, and what does that do to the forecast for each month of the year?

*Your answers here:*

---

## Part 2: FIX — Correct the Pipeline (24 min)

Repair the three steps, keeping the analyst's AR and MA orders, (2, d, 1).

**Reading the code.** `SARIMAX(series, order=(p, d, q), seasonal_order=(P, D, Q, m))` is an ARIMA with
a seasonal part of period m: D seasonal differences (each month minus the same month a year
earlier), P seasonal AR terms and Q seasonal MA terms. For the seasonal part, use the one in the
chapter's SARIMA for CPI (Section 21.2, "Code: SARIMA on CPI"). `.fit(disp=False)` fits it
without printing the optimiser's progress.

Each difference needs past observations the series does not have at its start: one month for an
ordinary difference, m for a seasonal one. Until then statsmodels' "prediction" is a starting
guess, and it counts these start-up residuals in `result.loglikelihood_burn`. `result.resid.iloc[n:]`
keeps the residuals from position n on. A Ljung-Box test on a fitted model's residuals takes
`model_df`, the number of ARMA terms estimated: each one uses up a degree of freedom.

**Your task.** Fill in the three blanks. The second cell runs your repaired check on the random walk
first, so you see it reject before you read its result for your model.

In [ ]:
# YOUR TASK — Part 2: fill in the two blanks (____)
# Steps 1 and 2 repaired: the ADF decision, and the model with a seasonal part
stationary = ____                    # use adf_p from the analyst's Step 1
d = 0 if stationary else 1
seasonal_part = ____                 # (P, D, Q, m)
sarima = SARIMAX(cpi, order=(2, d, 1), seasonal_order=seasonal_part).fit(disp=False)
print(f"stationary: {stationary}   so d = {d}   start-up residuals: {sarima.loglikelihood_burn}")
print(sarima.params.round(4).to_string())

**Expected output:**
```
stationary: False   so d = 1   start-up residuals: 13
ar.L1       0.2028
ar.L2       0.0567
ma.L1       0.4172
ar.S.L12   -0.0281
ma.S.L12   -0.9931
sigma2      0.4164
```
`d = 0` means the first blank still gives `True` for a p-value of 0.9571. No `ar.S.L12` row (and `ma.S.L12` at −0.9978) means a seasonal part without the seasonal AR term, which is not the chapter's: the next cell then prints 0.0209 and 0.0227 for the SARIMA.

In [ ]:
# YOUR TASK — Part 2, continued: fill in the blank (____)
# Step 2 repaired: Ljung-Box on the residuals each model forecast, the random walk first
for name, result in [("random walk", random_walk), ("SARIMA", sarima)]:
    kept = ____                      # the residuals of result that are forecast errors
    model_df = len(result.params) - 1        # ARMA terms estimated: every parameter except sigma2
    p_values = acorr_ljungbox(kept, lags=[12, 24], model_df=model_df, return_df=True)["lb_pvalue"]
    print(f"{name:>11}: {len(kept)} residuals tested, Ljung-Box p-values at lags 12 and 24: "
          f"{p_values.iloc[0]:.4f} {p_values.iloc[1]:.4f}")
plot_acf(kept, lags=36)              # kept is SARIMA's now, the last model in the loop
plt.show()

**Expected output:**
```
random walk: 317 residuals tested, Ljung-Box p-values at lags 12 and 24: 0.0000 0.0000
     SARIMA: 305 residuals tested, Ljung-Box p-values at lags 12 and 24: 0.0144 0.0183
```
then the SARIMA residuals' autocorrelation chart. Random-walk p-values of 1.0000 mean the blank still keeps every residual (the SARIMA's are then 0.0000 and 0.0001). SARIMA p-values of 0.9998 and 1.0000, with 317 residuals tested, mean the blank leaves out one residual for every model; 305 residuals for the random walk mean a fixed 13. The SARIMA's start-up lasts 13 months, the random walk's one.

**Reading the code.** The next cell runs Step 3 again with the repaired model.

In [ ]:
# GIVEN — run as-is
# Step 3 with the repaired model: the 24 months after June 2026, with a 95% interval
sarima_forecast_24 = sarima.get_forecast(steps=24)
plot_forecast(sarima_forecast_24, "SARIMA forecast")
month_by_month(sarima_forecast_24)

**Expected output:** a chart, then
```
month   CPI's average change   forecast change in 2027
    1                  1.003                     0.999
    2                  1.069                     1.074
    3                  1.239                     1.259
    4                  0.834                     0.853
    5                  0.759                     0.774
    6                  0.755                     0.738
    7                  0.232                     0.220
    8                  0.332                     0.330
    9                  0.484                     0.482
   10                  0.063                     0.063
   11                 -0.396                    -0.397
   12                 -0.364                    -0.365
```

**One forecast, scored.** Box-Jenkins ends with a forecast; a forecast is judged by how far it
lands from what happened, in months it did not see. **MASE** (from class) divides the forecast's
mean absolute error by the mean absolute error of the *seasonal naive* forecast (each month equal
to the same month a year earlier) on the training data. Below 1, the forecast's errors are smaller
than the seasonal naive forecast's average error in sample; above 1, larger. The seasonal naive
forecast itself can score above 1 out of sample, as it does below.

**Reading the code.** `compute_mase` is that ratio, with `insample[m:] - insample[:-m]` the
in-sample seasonal naive errors for period m (each value minus the one m earlier); `np.asarray`
turns a Series or a list into plain numbers. `seasonal_naive(train, horizon)` repeats the last 12
months of `train` until there are `horizon` forecasts (`np.resize` does the repeating), and
`sarima_forecast(train, horizon)` fits your repaired model, with the orders read from `sarima`
(`sarima.model.order` and `sarima.model.seasonal_order`), to `train` and forecasts `horizon`
months. Both take the same two arguments and return `horizon` numbers, so either can be passed
around as `forecast_fn`. The second cell trains each on the months to June 2024 and scores its
forecast of July 2024 to June 2025. `.to_numpy()` gives a Series' plain numbers.

In [ ]:
# GIVEN — run as-is
# MASE, and two forecasters that take (train, horizon) and return horizon forecasts
def compute_mase(actual, forecast, insample, m=1):
    """MAE of the forecast, divided by the in-sample MAE of the seasonal naive forecast with period m."""
    actual, forecast, insample = np.asarray(actual), np.asarray(forecast), np.asarray(insample)
    scale = np.mean(np.abs(insample[m:] - insample[:-m]))
    return np.mean(np.abs(actual - forecast)) / scale


def seasonal_naive(train, horizon):
    """Each of the last 12 months of train, repeated until there are horizon forecasts."""
    return np.resize(train.to_numpy()[-12:], horizon)


def sarima_forecast(train, horizon):
    """Your repaired model, fitted to train, forecasting the horizon months after it."""
    model = SARIMAX(train, order=sarima.model.order, seasonal_order=sarima.model.seasonal_order)
    return model.fit(disp=False).forecast(horizon).to_numpy()

In [ ]:
# GIVEN — run as-is
# Train on the months to June 2024; score the forecasts of July 2024 to June 2025
train_2024 = cpi[:"2024-06-01"]
actual_2025 = cpi["2024-07-01":"2025-06-01"]
for name, forecast_fn in [("seasonal naive", seasonal_naive), ("SARIMA", sarima_forecast)]:
    forecast_12 = forecast_fn(train_2024, 12)
    mae = np.mean(np.abs(actual_2025.to_numpy() - forecast_12))
    print(f"{name:>14}: MAE {mae:.4f}   MASE {compute_mase(actual_2025, forecast_12, train_2024, m=12):.4f}")

**Expected output:**
```
seasonal naive: MAE 8.1609   MASE 1.3623
        SARIMA: MAE 1.8641   MASE 0.3112
```

### YOUR DIAGNOSIS, continued

1. Why does the random walk's line have to come first? Then read your SARIMA's p-values at 5%: is the repaired model done, what does the seasonal part fix (compare the month-by-month table with Part 1's), and where would Box-Jenkins send you next?
2. On this one year the seasonal naive forecast has a MASE above 1 and the SARIMA well below 1. Say what each number means, and why one year is not enough to choose between them.

*Your answers here:*

---

## Part 3: EXTEND — GARCH(1,1) on S&P 500 (15 min)

ARIMA models the **conditional mean** of a time series, and its interval assumes one shock
variance, `sigma2`, for every month. Financial returns show **volatility clustering**: large moves,
up or down, tend to follow large moves. GARCH(1,1) (Engle 1982, Bollerslev 1986, from class)
models the **conditional variance** as a function of yesterday's squared shock and yesterday's
variance:

$$\sigma_t^2 = \omega + \alpha_1 \epsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2$$

where $\alpha_1 + \beta_1 < 1$ keeps the variance stationary.

**Reading the code.** The data come from a saved copy in the course repository, so every number
below stays the same all term. The cell downloads the file's bytes (`urllib.request.urlopen`); if
that fails, `except` reads the copy in a downloaded course repository. `hashlib.md5(...)` is the
file's fingerprint: the cell stops unless it matches the saved copy's. `pd.read_csv(io.BytesIO(raw),
compression="gzip", index_col="date", parse_dates=True)` reads the bytes as a table dated by day.

In [ ]:
# GIVEN — run as-is
# The S&P 500's daily closes, January 2000 to December 2024, from the saved copy in the course repository
DATA_URL = ("https://raw.githubusercontent.com/RyanPiao/econ5200-student-notebooks/main/"
            "5200/data/lab-ch21-sp500-2026-10.csv.gz")
LOCAL_COPY = "data/lab-ch21-sp500-2026-10.csv.gz"
DATA_MD5 = "8058eaab3c059029261b837c078c5572"       # the fingerprint of the saved copy

try:
    with urllib.request.urlopen(DATA_URL, timeout=30) as response:
        raw = response.read()
except Exception:                    # works only in a downloaded copy of the course repository
    if not Path(LOCAL_COPY).exists():
        raise FileNotFoundError(
            "Could not download the S&P 500 file from the course repository, and there is no local "
            "copy. Check that Colab is online, then run this cell again.") from None
    raw = Path(LOCAL_COPY).read_bytes()

if hashlib.md5(raw).hexdigest() != DATA_MD5:
    raise ValueError(
        "The S&P 500 file is not the saved copy this lab was written for (its fingerprint differs), "
        "so your numbers would not match the expected outputs. Run this cell again; if the message "
        "stays, tell your instructor.")
sp500 = pd.read_csv(io.BytesIO(raw), compression="gzip", index_col="date", parse_dates=True)
print(f"{sp500.index[0].date()} to {sp500.index[-1].date()}: {len(sp500)} trading days")

**Expected output:** `2000-01-03 to 2024-12-30: 6288 trading days`.

**Reading the code.** The daily log return in percent is 100 × log(close today / close yesterday);
`.shift(1)` gives each row the day before. `.kurtosis()` is pandas' *excess* kurtosis, so a normal
distribution has 0.

In [ ]:
# GIVEN — run as-is
# Daily log returns, in percent, and their first four moments
close = sp500["close"]
returns = (np.log(close / close.shift(1)) * 100).dropna()
print(f"{len(returns)} returns: mean {returns.mean():.4f}%, std {returns.std():.4f}%, "
      f"skewness {returns.skew():.4f}, excess kurtosis {returns.kurtosis():.4f} (normal = 0)")

**Expected output:**
`6287 returns: mean 0.0223%, std 1.2235%, skewness -0.3862, excess kurtosis 10.4040 (normal = 0)`.

**Reading the code.** `arch_model(returns, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")`
is a constant mean plus GARCH(1,1) errors with normal shocks: in the `arch` library `p` is the
number of lagged squared shocks (the α terms) and `q` the number of lagged variances (the β terms).
`.fit(disp="off")` fits it without printing its progress. `.params` holds `mu` (the mean return), `omega`, `alpha[1]` and
`beta[1]`.

In [ ]:
# GIVEN — run as-is
# GARCH(1,1) with a constant mean and normal shocks
garch = arch_model(returns, mean="Constant", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
print(garch.params.round(4).to_string())
omega, alpha, beta = garch.params["omega"], garch.params["alpha[1]"], garch.params["beta[1]"]

**Expected output:**
```
mu          0.0615
omega       0.0235
alpha[1]    0.1197
beta[1]     0.8629
```

**Your task.** A shock that raises today's variance above its long-run level raises tomorrow's
expected variance by a share of that excess, and the day after's by the same share again: after k
days the share left is that share to the power k. Fill in the two blanks: `persistence`, the share
left after one day, from the estimates above, and `half_life`, the number of trading days k after
which half of the shock is left. `np.log` is the natural logarithm. The cell then prints the
long-run volatility the model implies beside the sample standard deviation.

In [ ]:
# YOUR TASK — Part 3: fill in the two blanks (____)
persistence = ____                   # the share of a variance shock left after one day
half_life = ____                     # trading days until half of the shock is left
print(f"persistence = {persistence:.4f}   half-life of a volatility shock: {half_life:.1f} trading days")
print(f"long-run volatility {np.sqrt(omega / (1 - persistence)):.4f}%   sample std {returns.std():.4f}%")

**Expected output:**
```
persistence = 0.9826   half-life of a volatility shock: 39.5 trading days
long-run volatility 1.1628%   sample std 1.2235%
```
A negative half-life means the logarithm of 2 stands where the logarithm of one half should be; 4.7 trading days means `persistence` holds β alone.

**Reading the code.** `.conditional_volatility` is the model's σ_t for each day. The dashed lines
mark the reopening after September 11, 2001, Lehman Brothers' bankruptcy, the COVID crash and the
June 2022 sell-off. `.idxmax()` is the date of the largest value.

In [ ]:
# GIVEN — run as-is
# The conditional volatility, with four days of stress marked
volatility = garch.conditional_volatility
stress_days = ["2001-09-17", "2008-09-15", "2020-03-16", "2022-06-13"]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(volatility, linewidth=0.6)
for day in stress_days:
    ax.axvline(pd.Timestamp(day), color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("conditional std (%)")
plt.show()
print(f"highest: {volatility.max():.4f}% on {volatility.idxmax().date()}; lowest: {volatility.min():.4f}%")
for day in stress_days:
    print(f"{day}: {volatility.loc[day]:.4f}%")

**Expected output:** a chart, then
```
highest: 6.8504% on 2020-03-17; lowest: 0.4462%
2001-09-17: 1.2803%
2008-09-15: 1.5613%
2020-03-16: 5.6154%
2022-06-13: 1.8079%
```

### Interpretation Questions

1. Which of α and β says how strongly volatility reacts to yesterday's return, and which says how long a high level lasts? Use your half-life to say how much of the March 2020 spike in variance was left a month later (about 21 trading days) and three months later (about 63).
2. Part 2's SARIMA interval for CPI assumes one shock variance, `sigma2`, for every month. If CPI's shocks clustered the way these returns do (2021–22, say), what would that do to the interval's width in calm years and in turbulent ones?

*Your answers here:*

---

## Write it yourself (required — 34 min)

This function is required. No AI: write it yourself.

Part 2 scored one forecast, from one origin. A forecaster that wins one year may lose the next, so
forecasts are compared over many origins, each using only the past: an *expanding-window backtest*
(from class). In the empty cell below, write

`backtest_expanding_window(series, forecast_fn, min_train, horizon=12, step=12, m=12)`

that scores `forecast_fn` at every origin and returns a table with one row per origin:

1. The origins are the positions `min_train`, `min_train + step`, `min_train + 2 * step`, and so on,
   as long as at least `horizon` points are left from the origin to the end of the series: the last
   one may be `len(series) - horizon`.
2. At each origin, `train` is every point before it, `series.iloc[:origin]`, a Series with its
   dates, so that the window grows by `step` each time; `actual` is the `horizon` points that start
   at the origin.
3. The forecast is `forecast_fn(train, horizon)`, as in Part 2's cell for one origin.
4. Record a dictionary with the keys `"train_end"` (the last date in `train`, `train.index[-1]`),
   `"n_train"` (the number of points in `train`), `"mae"` (the mean of the absolute errors,
   |actual − forecast|) and `"mase"` (`compute_mase(actual, forecast, train, m)`).
5. Return `pd.DataFrame(rows)`, where `rows` is the list of your dictionaries, as in Lab 20.

Start from Part 2's cell for one origin. You need about twelve lines.

The check runs your function on a made-up series with two settings of the arguments, with a
forecaster of its own that records what it was given, then on CPI. Until you write the function it
says "Not written yet"; after that it fails with a message until your function is right.

In [ ]:
# WRITE IT YOURSELF — required: write backtest_expanding_window(series, forecast_fn, min_train, horizon=12, step=12, m=12)

In [ ]:
# CHECK — for backtest_expanding_window above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "backtest_expanding_window" not in globals():
    print("Not written yet: write backtest_expanding_window in the empty cell above, then run this "
          "check again.")
else:
    import inspect

    def names_used(code):
        """Every global or attribute name a function's code refers to, including inner functions."""
        names = set(code.co_names)
        for const in code.co_consts:
            if hasattr(const, "co_names"):
                names |= names_used(const)
        return names

    def near(a, b):
        """Equal to about six significant digits."""
        return bool(np.isclose(a, b, rtol=1e-6, atol=1e-12))

    def all_near(got, want):
        """Two lists of numbers equal, entry by entry, to about six significant digits."""
        return len(got) == len(want) and all(near(a, b) for a, b in zip(got, want))

    COLUMNS = ["train_end", "n_train", "mae", "mase"]

    # inspect.signature lists the names of a function's arguments, in order, with their defaults
    signature = inspect.signature(backtest_expanding_window)
    params = list(signature.parameters)
    assert params[:3] == ["series", "forecast_fn", "min_train"], (
        f"backtest_expanding_window's arguments are ({', '.join(params)}). The check calls it as the "
        "task asks: backtest_expanding_window(series, forecast_fn, min_train, horizon=12, step=12, m=12).")
    for name in ["horizon", "step", "m"]:
        assert name in signature.parameters, (
            f"backtest_expanding_window has no argument {name}. Give it the arguments the task lists: "
            "(series, forecast_fn, min_train, horizon=12, step=12, m=12).")
        assert signature.parameters[name].default == 12, (
            f"Give {name} a default of 12, as the task says: Part 4 calls the function on CPI with "
            "only series, forecast_fn and min_train.")
    used = names_used(backtest_expanding_window.__code__)
    for lab_name in ["cpi", "analyst", "sarima", "random_walk", "train_2024", "actual_2025",
                     "forecast_12", "sp500", "returns"]:
        assert lab_name not in used, (
            f"backtest_expanding_window uses {lab_name}, a name from the lab, instead of its argument "
            "series. Compute everything from the arguments.")
    model_calls = {"seasonal_naive", "sarima_forecast", "ARIMA", "arch_model"} & used
    assert not model_calls, (
        f"backtest_expanding_window calls {', '.join(sorted(model_calls))} itself. Make every forecast "
        "with forecast_fn, the function it is given: the check hands it other forecasters.")
    assert "compute_mase" in used, (
        "backtest_expanding_window does not call compute_mase. Score each forecast with Part 2's "
        "compute_mase, so that every MASE in this lab is computed the same way.")

    calls = []

    def last_value(train, horizon):
        """A made-up forecaster: the last value of train, horizon times. It records how it was called."""
        has_dates = isinstance(train, pd.Series)
        calls.append({"series": has_dates, "n": len(train),
                      "end": train.index[-1] if has_dates else None, "horizon": horizon})
        return np.repeat(float(np.asarray(train)[-1]), horizon)

    def run(where, series, min_train, horizon, step, m):
        """Call the function on a copy; return its table, or stop with a message naming the cause."""
        calls.clear()
        given = series.copy()
        try:
            got = backtest_expanding_window(given, last_value, min_train, horizon, step, m)
        except ValueError as err:
            if calls and any(c["horizon"] != horizon for c in calls):
                raise AssertionError(
                    f"{where}, forecast_fn was called with horizon {calls[-1]['horizon']}, not "
                    f"{horizon}. Pass the function's own horizon argument: forecast_fn(train, horizon)."
                ) from None
            if calls and calls[-1]["n"] > len(series) - horizon:
                raise AssertionError(
                    f"{where}, there is an origin at position {calls[-1]['n']}, which has fewer than "
                    f"horizon = {horizon} points after it. The last origin is len(series) - horizon, "
                    "so the range must stop just after it.") from None
            raise AssertionError(
                f"{where}, the actual values and the forecast have different lengths ({err}). actual is "
                "the horizon values that start at the origin: series.iloc[origin:origin + horizon]."
            ) from None
        except TypeError as err:
            if "slice indexing" in str(err) or "DatetimeIndex" in str(err):
                raise AssertionError(
                    f"{where}, the function stopped with TypeError: {err}. .loc takes dates, and an "
                    "origin is a position: slice by position with .iloc.") from None
            if "required positional argument" in str(err):
                raise AssertionError(
                    f"{where}, the function stopped with TypeError: {err}. Call forecast_fn with two "
                    "arguments, as Part 2's cell did: forecast_fn(train, horizon).") from None
            raise AssertionError(f"{where}, the function stopped with TypeError: "
                                 f"{str(err).rstrip('.')}.") from None
        except AttributeError as err:
            if calls and not calls[-1]["series"]:
                raise AssertionError(
                    f"{where}, forecast_fn was given plain numbers, not a Series, so train lost its "
                    "dates. Slice the Series itself: series.iloc[:origin].") from None
            raise AssertionError(f"{where}, the function stopped with AttributeError: "
                                 f"{str(err).rstrip('.')}.") from None
        except Exception as err:
            raise AssertionError(f"{where}, the function stopped with {type(err).__name__}: "
                                 f"{str(err).rstrip('.')}.") from None
        assert given.equals(series), (
            f"{where}, the function changed the series it was given. Leave series as it is.")
        assert got is not None, f"{where}, the function gives back nothing: use return, not print."
        if isinstance(got, list):
            raise AssertionError(f"{where}, the function gives back a list. Return a table: "
                                 "pd.DataFrame(rows).")
        if not isinstance(got, pd.DataFrame):
            raise AssertionError(f"{where}, the function gives back a {type(got).__name__}, not a "
                                 "table. Return pd.DataFrame(rows).")
        missing = [c for c in COLUMNS if c not in got.columns]
        assert not missing, (
            f"{where}, the table has the columns {list(got.columns)}, and lacks {missing}. Give each "
            f"row's dictionary the keys {', '.join(COLUMNS)}, and do not make any of them the index.")
        return got.reset_index(drop=True)

    def check_windows(where, series, min_train, horizon, step):
        """The training windows forecast_fn was given, against the ones the task describes."""
        want = list(range(min_train, len(series) - horizon + 1, step))
        sizes = [c["n"] for c in calls]
        assert all(c["series"] for c in calls), (
            f"{where}, forecast_fn was given plain numbers, not a Series. Slice the Series itself: "
            "series.iloc[:origin].")
        assert all(c["horizon"] == horizon for c in calls), (
            f"{where}, forecast_fn was called with horizon {calls[0]['horizon']}, not {horizon}. Pass "
            "the function's own horizon argument: forecast_fn(train, horizon).")
        assert set(sizes) != {len(series)}, (
            f"{where}, forecast_fn was given the whole series every time. Give it only the points "
            "before the origin, train, so that no forecast sees the values it is scored on.")
        if len(sizes) > 1 and len(set(sizes)) == 1:
            raise AssertionError(
                f"{where}, every training window has {sizes[0]} points: the window rolls forward "
                "instead of growing. Each origin trains on everything before it: series.iloc[:origin].")
        WINDOW_SLIPS = [
            ([n + 1 for n in want], "each hold the origin itself, the first point the forecast is "
                                    "scored on. Train on series.iloc[:origin], which stops just before it"),
            (list(range(min_train - 1, len(series) - horizon + 1, step)),
             "start one point early. The first origin is min_train, so the first window has "
             "min_train points"),
            (want[1:], "skip the first origin. The first window is series.iloc[:min_train]"),
            (want[:-1], "stop one origin early. The last origin is len(series) - horizon, so the "
                        "range must run to len(series) - horizon + 1"),
            (list(range(min_train, len(series) - horizon + 1, 12)),
             "move forward 12 points at a time, not step. Use the step argument"),
        ]
        for slip, cause in WINDOW_SLIPS:
            assert sizes != slip or slip == want, (
                f"{where}, forecast_fn was given windows of {sizes} points. Your windows {cause}.")
        too_late = [n for n in sizes if n > len(series) - horizon]
        assert not too_late, (
            f"{where}, there is an origin at position {too_late[0]}, which has fewer than horizon = "
            f"{horizon} points after it, so its forecast is scored on fewer months. The last origin is "
            "len(series) - horizon: the range must run to len(series) - horizon + 1.")
        assert sizes == want, (
            f"{where}, forecast_fn was given windows of {sizes} points; they should be {want}. Compare "
            "each origin with the task: the first is min_train, the next ones step apart, the last "
            "len(series) - horizon.")

    def check_table(where, table, series, min_train, horizon, step, want_mae, want_mase, slips,
                    mae_slips=()):
        """The table's rows, dates, sizes and two scores, against values measured beforehand."""
        want = list(range(min_train, len(series) - horizon + 1, step))
        assert len(table) == len(want), (
            f"{where}, the table has {len(table)} row(s) for {len(want)} origins. Append one "
            "dictionary per origin, inside the loop.")
        assert pd.api.types.is_datetime64_any_dtype(table["train_end"]), (
            f"{where}, train_end holds values of type {type(table['train_end'].iloc[0]).__name__}, not "
            "pandas Timestamps: record train.index[-1] itself, not .date() or a string. Part 4 reads the "
            "column as dates.")
        ends = [pd.Timestamp(x) for x in table["train_end"]]
        assert ends != [series.index[n] for n in want], (
            f"{where}, train_end holds each origin's date, the first month forecast. Record the last "
            "date of train: train.index[-1].")
        assert ends == [series.index[n - 1] for n in want], (
            f"{where}, train_end is {[str(d.date()) for d in ends]}; it should be "
            f"{[str(series.index[n - 1].date()) for n in want]}, the last date of each train.")
        n_train = [int(x) for x in table["n_train"]]
        assert n_train == want, (
            f"{where}, n_train is {n_train}; it should be {want}, the number of points in each train.")
        mae = list(table["mae"])
        assert not all_near(mae, [x * horizon for x in want_mae]), (
            f"{where}, mae adds up the absolute errors. Take their mean: np.mean.")
        assert not any(x < 0 for x in mae), (
            f"{where}, some MAE values are negative: the errors were averaged with their signs. Take "
            "the absolute value of each error first, then the mean.")
        for slip, cause in mae_slips:
            assert not all_near(mae, slip), f"{where}, mae {cause}."
        assert all_near(mae, want_mae), (
            f"{where}, mae is {[round(x, 4) for x in mae]}; it should be "
            f"{[round(x, 4) for x in want_mae]}. MAE is the mean of the absolute errors, "
            "|actual - forecast|, over the horizon.")
        mase = list(table["mase"])
        assert not any(pd.isna(x) for x in mase), (
            f"{where}, some MASE values are missing (nan). compute_mase's third argument is the "
            "training data, train, whose seasonal naive errors scale the forecast's MAE.")
        for slip, cause in slips:
            assert not all_near(mase, slip), f"{where}, mase {cause}."
        assert all_near(mase, want_mase), (
            f"{where}, mase is {[round(x, 4) for x in mase]}; it should be "
            f"{[round(x, 4) for x in want_mase]}. Call compute_mase(actual, forecast, train, m).")

    # a made-up monthly series of 60 points: a trend, a yearly wave and noise
    made_up_gen = np.random.default_rng(2026)
    t = np.arange(60)
    made_up = pd.Series(np.round(100 + 0.5 * t + 3 * np.sin(2 * np.pi * t / 12)
                                 + made_up_gen.normal(0, 1, 60), 2),
                        index=pd.date_range("2019-01-01", periods=60, freq="MS"))
    M_IS_ONE = "is scaled by the one-step naive forecast (m = 1), compute_mase's default. Pass m on"
    WHOLE = ("is scaled by the naive errors of the whole series, including the months being forecast. "
             "Scale by train only")

    # 1. min_train=24, horizon=6, step=6, m=12
    where = "On a made-up series with min_train=24, horizon=6, step=6, m=12"
    table = run(where, made_up, 24, 6, 6, 12)
    check_windows(where, made_up, 24, 6, 6)
    check_table(where, table, made_up, 24, 6, 6,
                [5.605, 1.63, 4.431666667, 0.8666666667, 7.35, 2.563333333],
                [0.9633342882, 0.2708891146, 0.7187943502, 0.1429278215, 1.206841505, 0.4081740977],
                [([4.4300687285, 1.1185518221, 2.9880241444, 0.586165182, 5.0408580184, 1.7166624547],
                  M_IS_ONE),
                 ([0.9067133998, 0.2636829334, 0.7169048261, 0.1401995147, 1.1889997304, 0.4146670262],
                  WHOLE)],
                [([33.85645, 3.2494, 23.7560833333, 1.0281333333, 55.0146666667, 7.4026666667],
                  "is the mean of the squared errors. MAE is the mean of the absolute errors"),
                 ([5.605, 1.46, 4.4316666667, 0.4366666667, 7.35, 2.5633333333],
                  "is the absolute value of the mean error, so errors of opposite signs cancel. Take the "
                  "absolute value of each error first, then the mean")])

    # 2. min_train=19, horizon=6, step=5, m=3
    where = "On the same series with min_train=19, horizon=6, step=5, m=3"
    table = run(where, made_up, 19, 6, 5, 3)
    check_windows(where, made_up, 19, 6, 5)
    check_table(where, table, made_up, 19, 6, 5,
                [1.725, 5.605, 1.011666667, 4.516666667, 1.233333333, 2.816666667, 1.97, 2.563333333],
                [0.5655737705, 1.924227563, 0.313172203, 1.52092838, 0.4014103607, 0.9233495909,
                 0.6704646345, 0.8490063645],
                [([0.2846534653, 0.9633342882, 0.1674781705, 0.7317671895, 0.2033587786, 0.4637919797,
                   0.3192309377, 0.4081740977], "is scaled with m = 12, not the m it was given"),
                 ([1.3134517766, 4.4300687285, 0.6832288149, 3.129330254, 0.8174893889, 1.9197442807,
                   1.2793938574, 1.7166624547], M_IS_ONE),
                 ([0.5873655914, 1.9085125448, 0.344474313, 1.5379330944, 0.4199522103, 0.9590800478,
                   0.6707885305, 0.8728195938], WHOLE)])

    # 3. CPI and the seasonal naive forecast, with the defaults, as Part 4 calls it
    where = "On CPI with seasonal_naive and min_train=186"
    try:
        cpi_table = backtest_expanding_window(cpi.copy(), seasonal_naive, 186)
    except Exception as err:
        raise AssertionError(f"{where}, the function stopped with {type(err).__name__}: "
                             f"{str(err).rstrip('.')}.") from None
    assert isinstance(cpi_table, pd.DataFrame) and "mase" in cpi_table.columns and all_near(
        list(cpi_table["mase"]),
        [0.3441949492, 0.9776553297, 1.221860278, 1.134215178, 0.8675507664, 1.305411655,
         4.100452599, 3.353006046, 1.699157797, 1.362259772, 1.59063777]), (
        f"{where}, the function passed on the made-up series but gives other numbers here. Does it "
        "use anything other than its arguments and their defaults?")

    print("Passed: backtest_expanding_window matches on both made-up backtests and on CPI.")
    print("CPI, seasonal naive forecast, a 12-month horizon from each June:")
    print(cpi_table.round(4).to_string())

**Expected output:**
```
Passed: backtest_expanding_window matches on both made-up backtests and on CPI.
CPI, seasonal naive forecast, a 12-month horizon from each June:
    train_end  n_train      mae    mase
0  2015-06-01      186   1.6112  0.3442
1  2016-06-01      198   4.3827  0.9777
2  2017-06-01      210   5.4701  1.2219
3  2018-06-01      222   5.1421  1.1342
4  2019-06-01      234   3.9617  0.8676
5  2020-06-01      246   5.9207  1.3054
6  2021-06-01      258  18.8746  4.1005
7  2022-06-01      270  17.6598  3.3530
8  2023-06-01      282   9.8851  1.6992
9  2024-06-01      294   8.1609  1.3623
10 2025-06-01      306   9.6700  1.5906
```

---

## Part 4: Module Output — `forecast_evaluation.py` (12 min)

Here is the module, finished except for one function. It holds Part 2's `compute_mase` and
`seasonal_naive`, and a `sarima_forecaster` that builds a forecaster of the `(train, horizon)` kind
for any model orders, so that one backtest can compare several models.

**Your task.** Paste your `backtest_expanding_window` from Write it yourself, unchanged, at the end
of the first cell, where it says so. Then run the two cells, compare the output with the expected
output, and answer the questions under it. The first cell saves the module as a file; the second
imports it and backtests three forecasters on CPI.

**Reading the cells.** `%%writefile src/forecast_evaluation.py` saves the cell as that file instead
of running it. The notes after a colon or `->` (`horizon: int`, `-> float`) only say what type each
argument and result is. `sarima_forecaster` defines a function inside a function and returns it:
the inner `forecast` remembers the orders it was built with, and takes the same two arguments as
`seasonal_naive`. Its default orders are your repaired model's; `seasonal_order=(0, 0, 0, 0)` is no
seasonal part at all, which makes it the analyst's model with d = 1. In the second cell,
`sys.path.insert(0, "src")` lets Python find modules in the `src` folder, `importlib.reload` loads
the file again so that a re-run of the `%%writefile` cell takes effect, and `hasattr(...)` asks
whether the module defines a name. The three backtests
fit 22 models and take about ten seconds. `pd.DataFrame({...})` builds a table from a dictionary of
columns, and `.mean()` of a table is the mean of each column.

In [ ]:
%%writefile src/forecast_evaluation.py
"""Score time series forecasters by expanding-window backtest and MASE (ECON 5200 Lab 21).

A forecaster is a function forecast_fn(train, horizon) that returns horizon forecasts. Each is
scored at many forecast origins, trained only on the past. MASE scales its errors by the seasonal
naive forecast's in-sample errors: below 1, they are smaller.
"""
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX


def compute_mase(actual, forecast, insample, m: int = 1) -> float:
    """MAE of the forecast, divided by the in-sample MAE of the seasonal naive forecast with period m."""
    actual, forecast, insample = np.asarray(actual), np.asarray(forecast), np.asarray(insample)
    scale = np.mean(np.abs(insample[m:] - insample[:-m]))
    return np.mean(np.abs(actual - forecast)) / scale


def seasonal_naive(train: pd.Series, horizon: int) -> np.ndarray:
    """Each of the last 12 months of train, repeated until there are horizon forecasts."""
    return np.resize(train.to_numpy()[-12:], horizon)


def sarima_forecaster(order=(2, 1, 1), seasonal_order=(1, 1, 1, 12)):
    """A forecaster that fits a SARIMA with these orders to train and forecasts horizon points."""
    def forecast(train: pd.Series, horizon: int) -> np.ndarray:
        model = SARIMAX(train, order=order, seasonal_order=seasonal_order)
        return model.fit(disp=False).forecast(horizon).to_numpy()
    return forecast


# Paste your backtest_expanding_window function from Write it yourself below this line, unchanged.

In [ ]:
# GIVEN — run as-is, once your backtest_expanding_window is pasted into the module
# Import the module and backtest three forecasters on CPI, from June 2015 to June 2025
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import forecast_evaluation
importlib.reload(forecast_evaluation)        # reload, so a re-run of the %%writefile cell takes effect

if not hasattr(forecast_evaluation, "backtest_expanding_window"):
    print("Paste your backtest_expanding_window from Write it yourself into the module cell above "
          "first, then run that cell and this one again.")
else:
    forecasters = {"seasonal naive": forecast_evaluation.seasonal_naive,
                   "ARIMA(2,1,1)": forecast_evaluation.sarima_forecaster(seasonal_order=(0, 0, 0, 0)),
                   "SARIMA": forecast_evaluation.sarima_forecaster()}
    mase = {}
    for name, forecast_fn in forecasters.items():
        table = forecast_evaluation.backtest_expanding_window(cpi, forecast_fn, 186)
        mase[name] = table["mase"].to_numpy()
    by_origin = pd.DataFrame(mase, index=table["train_end"].dt.date)
    print(by_origin.round(4).to_string())
    print("mean MASE:")
    print(by_origin.mean().round(4).to_string())

**Expected output:**
```
            seasonal naive  ARIMA(2,1,1)  SARIMA
train_end                                       
2015-06-01          0.3442        0.2845  0.4841
2016-06-01          0.9777        0.3265  0.1146
2017-06-01          1.2219        0.6835  0.4301
2018-06-01          1.1342        0.3125  0.0559
2019-06-01          0.8676        0.2640  0.2937
2020-06-01          1.3054        0.8975  0.7338
2021-06-01          4.1005        1.7737  1.7982
2022-06-01          3.3530        0.5730  0.3181
2023-06-01          1.6992        0.5395  0.3130
2024-06-01          1.3623        0.5821  0.3112
2025-06-01          1.5906        0.5594  0.3382
mean MASE:
seasonal naive    1.6324
ARIMA(2,1,1)      0.6178
SARIMA            0.4719
```
`.dt.date` shows each date without its time of day. The backtest is not fully out of sample: d and
the seasonal orders were chosen on the whole series to June 2026, and the June 2025 origin is scored
against October 2025's filled-in value.

### YOUR DIAGNOSIS

1. Rank the three by mean MASE. In which origin does every forecaster do worst, what happened to inflation in the next twelve months, and would you report the mean alone?
2. The ARIMA(2,1,1) column is the analyst's model with only d repaired; the SARIMA column adds Part 2's seasonal part. What is the seasonal part worth, in mean MASE and in the number of origins the SARIMA wins?

*Your answers here:*

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR ...]` with your own words or the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed and repaired an analyst's ARIMA forecast of U.S. CPI
  (FRED CPIAUCNS, 2000 to June 2026): [YOUR THREE DIAGNOSES, ONE
  LINE EACH]; my repaired model's Ljung-Box p-values: [YOUR VALUES]
* Fitted a GARCH(1,1) to daily S&P 500 returns, 2000 to 2024:
  half-life of a volatility shock [YOUR VALUE] trading days
* Wrote an expanding-window backtest and shipped it in a module,
  forecast_evaluation.py; mean MASE over eleven origins:
  [YOUR VALUES FOR THE THREE FORECASTERS]

**Please write a README.md entry including:**
1. Project Title: Forecasting CPI and Market Volatility
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person, with no buzzwords. Use only
the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab21-forecasting`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab21-forecasting`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab21-forecasting`, branch `main`,
   commit message `Lab 21 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/forecast_evaluation.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.